# 07 - Cohere com Python

**Cohere** é uma empresa especializada em LLMs para uso empresarial, com foco em:
- **RAG** (Retrieval-Augmented Generation) com reranking nativo
- **Embeddings** multilíngues de alta qualidade
- **Command R+**: modelo conversacional para tarefas complexas

### Principais modelos

| Modelo | Uso recomendado |
|--------|-----------------|
| `command-r-plus` | Chat de alta qualidade, RAG |
| `command-r` | Rápido e econômico |
| `embed-multilingual-v3.0` | Embeddings multilíngues |
| `rerank-multilingual-v3.0` | Reranking de resultados |

### Pacotes necessários

```bash
pip install cohere python-dotenv
```

> Crie sua API key em [dashboard.cohere.com](https://dashboard.cohere.com) e adicione `COHERE_API_KEY=sua_chave` no `.env`.

> **Atenção**: a versão 5+ do SDK Cohere tem mudanças de API em relação às versões anteriores.
> `client.chat()` foi substituído por `client.chat.message()` ou a classe `ClientV2`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import cohere

load_dotenv(find_dotenv())

api_key = os.environ.get("COHERE_API_KEY")

# ClientV2: interface moderna do SDK Cohere (v5+)
client = cohere.ClientV2(api_key=api_key)

## 1. Chat básico

O SDK v2 do Cohere usa o padrão OpenAI de mensagens com `role` e `content`.

> **Mudança importante**: o antigo `client.chat(message=...)` foi substituído por
> `client.chat(messages=[...])` com lista de mensagens no SDK v2.

In [ ]:
response = client.chat(
    model="command-r-plus",
    messages=[
        {"role": "user", "content": "Oi. Quem é você?"}
    ]
)

print(response.message.content[0].text)

## 2. Streaming

Com `client.chat_stream()`, os eventos chegam progressivamente.
O Cohere usa `event_type` para distinguir entre diferentes tipos de eventos no stream.

| `event_type` | Significado |
|--------------|-------------|
| `"content-delta"` | Fragmento de texto gerado (SDK v2) |
| `"message-end"` | Fim da mensagem |

In [ ]:
# SDK v2: stream via client.chat_stream com messages=[...]
stream = client.chat_stream(
    model="command-r-plus",
    messages=[
        {"role": "user", "content": "Oi. Quem é você?"}
    ]
)

for event in stream:
    # content-delta contém os fragmentos de texto
    if event.type == "content-delta":
        print(event.delta.message.content.text, end="", flush=True)

## 3. Histórico de Conversa

O Cohere mantém o histórico da mesma forma que a OpenAI — passando todas as mensagens anteriores
em cada nova chamada.

Você controla o histórico manualmente, adicionando as mensagens do usuário e do assistente.

In [ ]:
# Histórico de conversa — você gerencia manualmente a lista de mensagens
historico = [
    {"role": "user",      "content": "Oi, eu sou Rodrigo!"},
    {"role": "assistant", "content": "Olá, Rodrigo! Como posso lhe ajudar?"}
]

# Nova pergunta — o modelo tem contexto das mensagens anteriores
historico.append({"role": "user", "content": "Qual meu nome?"})

stream = client.chat_stream(
    model="command-r-plus",
    messages=historico
)

for event in stream:
    if event.type == "content-delta":
        print(event.delta.message.content.text, end="", flush=True)

## Resumo

| Funcionalidade | Método |
|---------------|--------|
| Chat simples | `client.chat(model=..., messages=[...])` |
| Streaming | `client.chat_stream(...)` + filtrar `event.type == "content-delta"` |
| Histórico | passar todas as mensagens na lista `messages` |

> **Diferencial do Cohere**: excelente para **RAG empresarial** — possui modelos de reranking
> e embeddings multilíngues que melhoram significativamente a qualidade da busca semântica.